In [3]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, accuracy_score, roc_curve, auc, precision_recall_curve, ConfusionMatrixDisplay
from sklearn.preprocessing import label_binarize
from matplotlib.backends.backend_pdf import PdfPages
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Loss
import tensorflow as tf

# === Configuración ===
INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"
OUTPUT_BASE = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\models"
CATEGORIA = "Standard_FE_ALL"
INTENTO = 1
fecha_actual = datetime.today().strftime('%Y-%m-%d')
OUTPUT_PATH = os.path.join(OUTPUT_BASE, f"KerasSMOTE_{INTENTO:02d}_{CATEGORIA}_{fecha_actual}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# === Leer archivos ===
X_train = pd.read_parquet(os.path.join(INPUT_PATH, f"X_train_{CATEGORIA}.parquet"))
X_test = pd.read_parquet(os.path.join(INPUT_PATH, f"X_test_{CATEGORIA}.parquet"))
y_train = pd.read_parquet(os.path.join(INPUT_PATH, f"y_train_{CATEGORIA}.parquet")).squeeze()
y_test = pd.read_parquet(os.path.join(INPUT_PATH, f"y_test_{CATEGORIA}.parquet")).squeeze()

# === Aplicar SMOTE al train ===
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# === Focal Loss personalizada ===
class FocalLoss(Loss):
    def __init__(self, gamma=2., alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = self.alpha * tf.math.pow(1 - y_pred, self.gamma)
        return tf.reduce_sum(weight * cross_entropy, axis=1)

# === Modelo Keras ===
num_features = X_train.shape[1]
num_classes = np.max(y_train_res) + 1
y_train_cat = to_categorical(y_train_res, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

model = Sequential([
    Dense(128, input_dim=num_features, activation='relu'),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(loss=FocalLoss(), optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

start = time.time()
history = model.fit(X_train_res, y_train_cat, epochs=30, batch_size=64, verbose=1, validation_split=0.1)
tiempo = (time.time() - start) / 60

# === Predicciones ===
y_pred_proba = model.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)

# === Reporte ===
classes = np.unique(y_test)
report_test = classification_report(y_test, y_pred, output_dict=True)
df_report = pd.DataFrame(report_test).T
df_report['tiempo_min'] = tiempo
df_report.to_csv(os.path.join(OUTPUT_PATH, f"metricas_{CATEGORIA}.csv"))

# === PDF ===
y_test_bin = label_binarize(y_test, classes=classes)

with PdfPages(os.path.join(OUTPUT_PATH, f"reporte_{CATEGORIA}.pdf")) as pdf:
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues')
    plt.title("Matriz de Confusión")
    pdf.savefig(); plt.close()

    for i, clase in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
        auc_score = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"Clase {clase} (AUC={auc_score:.2f})")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title("Curvas ROC"); plt.legend(); pdf.savefig(); plt.close()

    for i, clase in enumerate(classes):
        precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_pred_proba[:, i])
        plt.plot(recall, precision, label=f"Clase {clase}")
    plt.title("Curvas Precision-Recall"); plt.legend(); pdf.savefig(); plt.close()

print(f"✅ Entrenamiento finalizado. Tiempo total: {tiempo:.2f} min")


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
8745/8745 ━━━━━━━━━━━━━━━━━━━━ 33s 3ms/step - accuracy: 0.4076 - loss: 0.2338 - val_accuracy: 0.1872 - val_loss: 0.2058
Epoch 2/30
8745/8745 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - accuracy: 0.4543 - loss: 0.1700 - val_accuracy: 0.2577 - val_loss: 0.1901
Epoch 3/30
8745/8745 ━━━━━━━━━━━━━━━━━━━━ 26s 3ms/step - accuracy: 0.4619 - loss: 0.1668 - val_accuracy: 0.2740 - val_loss: 0.1994
Epoch 4/30
8745/8745 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - accuracy: 0.4626 - loss: 0.1661 - val_accuracy: 0.1896 - val_loss: 0.2036
Epoch 5/30
8745/8745 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - accuracy: 0.4639 - loss: 0.1646 - val_accuracy: 0.1957 - val_loss: 0.2012
Epoch 6/30
8745/8745 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - accuracy: 0.4646 - loss: 0.1640 - val_accuracy: 0.2115 - val_loss: 0.1953
Epoch 7/30
8745/8745 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - accuracy: 0.4648 - loss: 0.1636 - val_accuracy: 0.2022 - val_loss: 0.1987
Epoch 8/30
8745/8745 ━━━━━━━━━━━━━━━━━━━━ 24s 3ms/step - accuracy: 0.4668 - loss: 0